<a href="https://colab.research.google.com/github/yc-115/programing-language/blob/main/%E3%80%8CHW4_PTT_GoogleSheet_RAG%E6%95%B4%E7%90%86%E7%89%88_ipynb%E3%80%8D41371211H.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 安裝必要套件
!pip -q install gspread gspread-dataframe faiss-cpu sentence-transformers beautifulsoup4 requests google-generativeai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 43.9 MB/s eta 0:00:00


In [2]:
import re
import time
import uuid
from datetime import datetime
from urllib.parse import urljoin

import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup

import faiss
from sentence_transformers import SentenceTransformer

import google.generativeai as genai
from google.colab import auth, userdata
import gspread
from google.auth import default
from gspread_dataframe import set_with_dataframe, get_as_dataframe


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [3]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1CJOIag6uo5U_jXCEIuJvSZW89-pWk0Ht9lb-6VZcrNc/edit?usp=sharing"
PTT_WORKSHEET_NAME = "ptt_movie_posts"
TIMEZONE_NOTE = "Asia/Taipei"

PTT_HEADER = [
    "post_id", "title", "url", "date", "author", "nrec",
    "created_at", "fetched_at", "content"
]

PTT_MOVIE_INDEX = "https://www.ptt.cc/bbs/movie/index.html"
PTT_COOKIES = {"over18": "1"}
USER_AGENT = "Mozilla/5.0 (compatible; Colab PTT crawler)"


In [4]:
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# 關鍵修正：直接用網址開啟指定 Google Sheet
sh = gc.open_by_url(SHEET_URL)
print(f"✅ 已開啟試算表：{sh.title}")
print(f"🔗 {SHEET_URL}")


✅ 已開啟試算表：程式語言作業範例測試資料 41371211H
🔗 https://docs.google.com/spreadsheets/d/1CJOIag6uo5U_jXCEIuJvSZW89-pWk0Ht9lb-6VZcrNc/edit?usp=sharing


In [5]:
def ensure_worksheet(spreadsheet, title, header, rows=1000):
    """取得或建立 worksheet，並確保表頭正確。"""
    try:
        ws = spreadsheet.worksheet(title)
    except gspread.WorksheetNotFound:
        ws = spreadsheet.add_worksheet(title=title, rows=str(rows), cols=str(len(header) + 5))
        ws.update([header])
        return ws

    values = ws.get_all_values()
    if not values:
        ws.update([header])
    elif values[0] != header:
        # 保留資料但重建欄位較危險，因此這裡直接清掉並重新建立正確表頭。
        # 若你要保留舊資料，請先備份 Google Sheet。
        ws.clear()
        ws.update([header])
    return ws


def read_sheet_df(ws, header):
    """從 worksheet 讀成 DataFrame，並清掉空列。"""
    df = get_as_dataframe(ws, evaluate_formulas=True, dtype=str).dropna(how="all")
    if df.empty:
        return pd.DataFrame(columns=header)
    df = df.loc[:, [c for c in df.columns if not str(c).startswith("Unnamed")]]
    for col in header:
        if col not in df.columns:
            df[col] = ""
    return df[header].fillna("")


def write_sheet_df(ws, df, header):
    """把 DataFrame 寫回 worksheet。"""
    df_out = df.copy()
    for col in header:
        if col not in df_out.columns:
            df_out[col] = ""
    df_out = df_out[header].infer_objects(copy=False).fillna("") # Add infer_objects to address FutureWarning

    # Google Sheet 寫入前統一轉字串，避免 Timestamp / NaN 型別問題
    for c in df_out.columns:
        df_out[c] = df_out[c].astype(str)

    ws.clear()
    set_with_dataframe(ws, df_out, include_index=False, include_column_header=True, resize=True)
    return len(df_out)


ws_ptt = ensure_worksheet(sh, PTT_WORKSHEET_NAME, PTT_HEADER)
print(f"✅ 已準備 worksheet：{ws_ptt.title}")

✅ 已準備 worksheet：ptt_movie_posts


In [6]:
def now_iso():
    return datetime.now().isoformat(timespec="seconds")


def get_soup(url):
    resp = requests.get(
        url,
        timeout=20,
        headers={"User-Agent": USER_AGENT},
        cookies=PTT_COOKIES,
    )
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")


def get_prev_index_url(soup):
    for a in soup.select("div.btn-group-paging a.btn.wide"):
        if "上頁" in a.get_text(strip=True):
            href = a.get("href")
            return urljoin("https://www.ptt.cc", href) if href else None
    return None


def parse_nrec(nrec_span):
    if not nrec_span:
        return 0
    txt = nrec_span.get_text(strip=True)
    if txt == "爆":
        return 100
    if txt.startswith("X"):
        try:
            return -int(txt[1:])
        except Exception:
            return -10
    try:
        return int(txt)
    except Exception:
        return 0


def extract_post_list(index_soup):
    posts = []
    for item in index_soup.select("div.r-ent"):
        a = item.select_one("div.title a")
        if not a:
            continue

        title = a.get_text(strip=True)
        url = urljoin("https://www.ptt.cc", a.get("href"))
        author_node = item.select_one("div.author")
        date_node = item.select_one("div.date")
        nrec_node = item.select_one("div.nrec span")

        posts.append({
            "title": title,
            "url": url,
            "author": author_node.get_text(strip=True) if author_node else "",
            "date": date_node.get_text(strip=True) if date_node else "",
            "nrec": parse_nrec(nrec_node),
        })
    return posts


def clean_ptt_content(article_soup):
    main = article_soup.select_one("#main-content")
    if not main:
        return "", ""

    # 取出文章建立時間
    created_at = ""
    metalines = main.select("div.article-metaline")
    for m in metalines:
        tag = m.select_one("span.article-meta-tag")
        value = m.select_one("span.article-meta-value")
        if tag and value and tag.get_text(strip=True) == "時間":
            created_at = value.get_text(strip=True)

    # 移除 meta 與推文
    for node in main.select("div.article-metaline, div.article-metaline-right, div.push"):
        node.decompose()

    text = main.get_text("\n", strip=True)
    text = re.split(r"\n--\n|\n※ 發信站:", text)[0].strip()
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text, created_at


def make_post_id(url):
    # 用文章網址檔名當 post_id，穩定且方便去重
    return url.rstrip("/").split("/")[-1].replace(".html", "")


def crawl_ptt_movie(pages=2, delay=0.5):
    """爬取 PTT movie 最新 pages 頁文章。"""
    all_rows = []
    index_url = PTT_MOVIE_INDEX

    for page in range(int(pages)):
        print(f"📄 正在讀取列表頁 {page + 1}/{pages}: {index_url}")
        index_soup = get_soup(index_url)
        post_list = extract_post_list(index_soup)

        for p in post_list:
            try:
                article_soup = get_soup(p["url"])
                content, created_at = clean_ptt_content(article_soup)
                row = {
                    "post_id": make_post_id(p["url"]),
                    "title": p["title"],
                    "url": p["url"],
                    "date": p["date"],
                    "author": p["author"],
                    "nrec": p["nrec"],
                    "created_at": created_at,
                    "fetched_at": now_iso(),
                    "content": content,
                }
                all_rows.append(row)
                time.sleep(delay)
            except Exception as e:
                print(f"⚠️ 跳過文章：{p.get('title', '')}，原因：{e}")

        prev_url = get_prev_index_url(index_soup)
        if not prev_url:
            break
        index_url = prev_url
        time.sleep(delay)

    df = pd.DataFrame(all_rows, columns=PTT_HEADER)
    print(f"✅ 本次爬到 {len(df)} 篇文章")
    return df

In [8]:
# 你可以調整 pages，例如 pages=1 先測試，確認成功後再改成 3 或 5
new_posts_df = crawl_ptt_movie(pages=2, delay=1.0)

old_posts_df = read_sheet_df(ws_ptt, PTT_HEADER)
print(f"📌 Google Sheet 原本有 {len(old_posts_df)} 筆")

ptt_posts_df = pd.concat([old_posts_df, new_posts_df], ignore_index=True)
ptt_posts_df = ptt_posts_df.drop_duplicates(subset=["post_id"], keep="last")
ptt_posts_df = ptt_posts_df.sort_values(by="fetched_at", ascending=False)

written_count = write_sheet_df(ws_ptt, ptt_posts_df, PTT_HEADER)
print(f"✅ 已寫入 Google Sheet：{written_count} 筆")

verify_df = read_sheet_df(ws_ptt, PTT_HEADER)
print(f"🔍 從 Google Sheet 重新讀回：{len(verify_df)} 筆")

if len(verify_df) == written_count:
    print("✅ 寫入驗證成功")
else:
    print("⚠️ 寫入筆數與讀回筆數不同，請檢查 Google Sheet 權限或資料格式")

📄 正在讀取列表頁 1/2: https://www.ptt.cc/bbs/movie/index.html
📄 正在讀取列表頁 2/2: https://www.ptt.cc/bbs/movie/index11001.html
⚠️ 跳過文章：[ 好雷] 給阿嬤的情書，原因：('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
✅ 本次爬到 26 篇文章
📌 Google Sheet 原本有 56 筆
✅ 已寫入 Google Sheet：57 筆
🔍 從 Google Sheet 重新讀回：57 筆
✅ 寫入驗證成功


In [9]:
# 從 Google Sheet 重新讀取，作為 RAG 的唯一資料來源
rag_source_df = read_sheet_df(ws_ptt, PTT_HEADER)

# 清掉沒有內容的文章
rag_source_df = rag_source_df[rag_source_df["content"].astype(str).str.strip() != ""].copy()
print(f"📚 可用於 RAG 的文章數：{len(rag_source_df)}")

rag_source_df.head()


📚 可用於 RAG 的文章數：57


,post_id,title,url,date,author,nrec,created_at,fetched_at,content
0,M.1780335413.A.1E1,[普無雷] 《出口成髒》不建議看我這篇,https://www.ptt.cc/bbs/movie/M.1780335413.A.1E...,6/2,nobady98,0,Tue Jun 2 01:36:51 2026,2026-06-02 3:15:12,上週我趕不上最後一場\n以為沒得看了\n沒想到那只是口碑場\n\n劇情有點太過跳躍\n節奏忽...
1,M.1780334467.A.A2A,[討論] DCU 雷克斯路瑟動力裝甲片廠照,https://www.ptt.cc/bbs/movie/M.1780334467.A.A2...,6/2,labich,15,Tue Jun 2 01:21:05 2026,2026-06-02 3:15:10,http://i.imgur.com/PGVbCQs.jpg\nhttps://x.com/...
2,M.1780328198.A.BB9,[新聞]拍《明日邊界》艾蜜莉布朗讚阿湯哥不一般！,https://www.ptt.cc/bbs/movie/M.1780328198.A.BB...,6/1,XDGEE,7,Mon Jun 1 23:36:36 2026,2026-06-02 3:15:08,拍《明日邊界》超折磨！艾蜜莉布朗讚阿湯哥不一般！\n最近，英國女演員艾蜜莉‧布朗回憶起在拍《...
3,M.1780328150.A.D43,[新聞]史匹柏談創意底線：不能讓AI成為決策者！,https://www.ptt.cc/bbs/movie/M.1780328150.A.D4...,6/1,XDGEE,2,Mon Jun 1 23:35:48 2026,2026-06-02 3:15:06,匹柏談創意底線：不能讓AI成為決策者！\n隨著好萊塢關於人工智慧（AI）的應用議題持續升溫，...
4,M.1780326785.A.E3B,[討論]安海瑟薇、伊旺麥奎格《橡樹街末日》預告！,https://www.ptt.cc/bbs/movie/M.1780326785.A.E3...,6/1,XDGEE,5,Mon Jun 1 23:13:03 2026,2026-06-02 3:15:05,安海瑟薇、伊旺麥奎格《橡樹街末日》預告！\nhttps://youtu.be/dqUZZ9k...


In [10]:
print("正在載入多語言 Embedding 模型...")
embedding_model = SentenceTransformer("BAAI/bge-m3")
print("✅ Embedding 模型載入完成")


def build_rag_documents(df):
    docs = []
    for _, row in df.iterrows():
        title = str(row.get("title", ""))
        content = str(row.get("content", ""))
        url = str(row.get("url", ""))
        author = str(row.get("author", ""))
        date = str(row.get("date", ""))
        nrec = str(row.get("nrec", ""))

        text = (f"標題：{title}\n"
                f"作者：{author}\n"
                f"日期：{date}\n"
                f"推文數：{nrec}\n"
                f"內容：{content}")
        docs.append({
            "post_id": str(row.get("post_id", "")),
            "title": title,
            "url": url,
            "text": text,
        })
    return docs


def build_faiss_index(docs):
    if not docs:
        raise ValueError("沒有可建立索引的文件")

    texts = [d["text"] for d in docs]
    embeddings = embedding_model.encode(texts, convert_to_numpy=True, show_progress_bar=True)
    embeddings = embeddings.astype("float32")

    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)
    return index, embeddings


rag_documents = build_rag_documents(rag_source_df)
rag_index, rag_embeddings = build_faiss_index(rag_documents)

print(f"✅ RAG 索引建立完成：{len(rag_documents)} 篇文章，向量維度 {rag_embeddings.shape[1]}")

正在載入多語言 Embedding 模型...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

✅ Embedding 模型載入完成


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

✅ RAG 索引建立完成：57 篇文章，向量維度 1024


In [11]:
api_key = userdata.get("gemini")
if not api_key:
    raise ValueError("找不到 Colab Secret：gemini。請先在 Colab Secrets 新增 Gemini API key。")

genai.configure(api_key=api_key)

# 若你的帳號不支援這個模型，可改成你可用的 Gemini model name
GEMINI_MODEL_NAME = "gemini-3-flash-preview"
llm = genai.GenerativeModel(GEMINI_MODEL_NAME)
print(f"✅ Gemini 已設定：{GEMINI_MODEL_NAME}")


✅ Gemini 已設定：gemini-3-flash-preview


In [12]:
def retrieve_docs(query, k=3):
    if rag_index is None or not rag_documents:
        return []
    q_emb = embedding_model.encode([query], convert_to_numpy=True).astype("float32")
    distances, indices = rag_index.search(q_emb, int(k))

    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx == -1:
            continue
        doc = rag_documents[idx].copy()
        doc["distance"] = float(dist)
        results.append(doc)
    return results


def query_rag(question, k=3):
    docs = retrieve_docs(question, k=k)
    if not docs:
        return "找不到相關 PTT 資料。"

    context = "\n\n---\n\n".join(
        [f"來源標題：{d['title']}\n來源網址：{d['url']}\n{d['text']}" for d in docs]
    )

    prompt = f"""
你是一個根據 PTT 電影版資料回答問題的助教。
請只根據【PTT 資料】回答。
如果資料不足，請明確說「目前資料不足，無法判斷」，不要自行編造。
回答請使用繁體中文，並在最後列出參考來源標題與網址。

【PTT 資料】
{context}

【問題】
{question}

【回答】
""".strip()

    response = llm.generate_content(prompt)
    return response.text

In [13]:
question = input("請輸入問題：")
answer = query_rag(question, k=3)
print(answer)


請輸入問題：2026 年剛公佈的電影消息
根據【PTT 資料】，2026 年 6 月公佈與討論的相關電影消息如下：

1. **YouTuber 執導電影票房大捷**：
   在 2026 年 6 月初的消息指出，由兩位才 20 出頭的 YouTuber 執導的電影《後室》（後室）與《愛你至死不渝》（愛你至死不渝），在過去兩週內轟動全世界，票房雙雙突破億元大關。這被認為是主流大廠願意給予年輕創作者機會的新趨勢。

2. **台灣 YouTuber 影壇動向討論**：
   * 網友提及曾有 YouTuber 進入電影圈的案例，例如陳奕凱（樂咖）曾於 2019 年以作品《偷偷》入圍第 56 屆金馬獎，但其頻道目前處於停擺狀態。
   * 針對「反正我很閒」是否有機會拍電影，有討論認為雖然他們本來機會很大，但因其具備「藝術家個性」，要其積極製作電影似乎不切實際。

3. **近期電影評論**：
   2026 年 6 月亦有針對 2025 年電影《生命清單》（The Life List）的討論。該片講述女主角 Alex 執行母親遺囑中的兒時願望清單，故事涉及脫口秀、彈琴、與父親和好及尋找真愛等任務，最終結局溫馨。

參考來源：
* Re: [討論] 台灣什麼時候能出現YouTuber拍電影的八 (https://www.ptt.cc/bbs/movie/M.1780346942.A.DE3.html)
* [討論] 台灣什麼時候能出現YouTuber拍電影的八 (https://www.ptt.cc/bbs/movie/M.1780322964.A.920.html)
* [好雷] The Life List (2025) 生命清單 (https://www.ptt.cc/bbs/movie/M.1780356684.A.6E9.html)


In [14]:
!pip install -q gradio

In [15]:
import gradio as gr

# 1. 定義 Gradio 的核心包裹函式
def rag_chat_interface(question, k_slider):
    """
     question: 使用者輸入的問題
     k_slider: 使用者在介面上調整的 K 值（要搜尋幾篇 PTT 文章）
    """
    if not question.strip():
        return "請輸入問題後再點擊送出喔！"

    try:
        # 呼叫你原本寫好的 RAG 查詢函式，並帶入介面上選擇的 k 值
        answer = query_rag(question, k=int(k_slider))
        return answer
    except Exception as e:
        return f"❌ 系統發生錯誤：{str(e)}"

# 2. 建立 Gradio 美化介面
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        # 🎬 PTT 電影版 RAG 智慧問答系統
        ### 💡 HW4 實作展示 —— 基於最新 PTT 電影版資料與進階 Embedding 模型
        """
    )

    with gr.Row():
        with gr.Column(scale=3):
            # 左側：輸入問題與調整參數
            user_input = gr.Textbox(
                label="請輸入您的問題",
                placeholder="例如：2026 年剛公佈的電影消息、有沒有阿湯哥電影的評價？",
                lines=3
            )
            # 讓使用者可以動態調整 K 值（作業核心亮點！）
            k_value = gr.Slider(
                minimum=1,
                maximum=5,
                value=3,
                step=1,
                label="🔍 檢索文章數量 (K 值)"
            )
            submit_btn = gr.Button("🚀 點我詢問 AI 助教", variant="primary")

        with gr.Column(scale=4):
            # 右側：顯示 AI 的回答與參考來源
            output_display = gr.Textbox(
                label="🤖 AI 助教根據 PTT 資料的回答",
                lines=15,
                interactive=False
            )

    # 設定按鈕點擊事件
    submit_btn.click(
        fn=rag_chat_interface,
        inputs=[user_input, k_value],
        outputs=output_display
    )

# 3. 啟動 Gradio（在 Colab 中必須設定 share=True）
demo.launch(share=True, debug=True)

/tmp/ipykernel_1960/3600541643.py:20: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://26cd09460b9a8f991e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://26cd09460b9a8f991e.gradio.live
